# Random forest classifier for written numbers

In [97]:
from random import random
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, ParameterGrid
import numpy as np
from time import perf_counter
import numpy as np
from RandomForest import RandomForest
import pandas as pd

## Data preprocessing

In [98]:
file_path = "D:\Github repos\PRML_project\SVM\parquets"
df_dict = []
for file in range(0,10):
  df = pd.read_parquet(f'{file_path}/Num_{file}.parquet')
  grouped = df.groupby(level='sheet')
  df_dict.append(grouped)
# every file is now found in the dict

def interpolate_data(points, target_length):
  #cumulative arc length
  diffs = np.diff(points, axis=0)
  segment_len = np.sqrt(np.sum(diffs**2, axis=1))
  cumlen = np.concatenate(([0], np.cumsum(segment_len)))
  total_len = cumlen[-1]
  # create target archs
  target_arc_len = np.linspace(0, total_len, target_length)
  # interpolate
  interpolated = np.zeros((target_length, points.shape[1]))
  for coord  in range(points.shape[1]):
    interpolated[:, coord] = np.interp(target_arc_len, cumlen, points[:, coord])
  # resample
  return interpolated

# interpolate each number
number_dict = {}
for number in range(10):
  target = 55 # global average
  key = str(number)
  interp_numbers = []
  for sample in range(1,101):
    study_df = df_dict[number]
    sample_df = study_df.get_group(sample)
    # call the interpolation
    intp_data1 = interpolate_data(sample_df.to_numpy(), target)
    interp_numbers.append(intp_data1)
  # add the dictionary
  number_dict[key] = interp_numbers

# So there are 10 numbers:
print(len(number_dict['3']))
# Each number has 100 samples
print(number_dict['3'][0].shape)
# and each sample has 55 points with 3 columns, so 165 features per sample

# reshaping the data to be X and y arrays
X = []
y = []
for number in range(10):
  key = str(number)
  for sample in range(100):
    X.append(number_dict[key][sample].flatten())
    y.append(number)
X = np.array(X)
y = np.array(y)
print(X.shape)
print(y.shape)

100
(55, 3)
(1000, 165)
(1000,)


## Model

In [ ]:
def accuracy(y_true, y_pred):
    accuracy = np.sum(y_true == y_pred) / len(y_true)
    return accuracy


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

cv = KFold(n_splits=5, shuffle=True, random_state=0)

folds = []

for i, (train_idx, val_idx) in enumerate(cv.split(X_train), 1):
    print(f"Fold {i}: train={len(train_idx)} val={len(val_idx)}")
    folds.append((train_idx, val_idx))

accs = []
for train_idx, val_idx in folds:
    clf = RandomForest(n_trees=30, max_depth=30, min_samples_split=2, n_feature=26)
    clf.fit(X_train[train_idx], y_train[train_idx])
    predictions = clf.predict(X_train[val_idx])
    accs.append(accuracy(y_train[val_idx], predictions))

Fold 1: train=640 val=160
Fold 2: train=640 val=160
Fold 3: train=640 val=160
Fold 4: train=640 val=160
Fold 5: train=640 val=160


In [127]:
accs = pd.Series(accs)
print(accs)
print(accs.mean())

0    0.90625
1    0.93750
2    0.92500
3    0.93125
4    0.95000
dtype: float64
0.9299999999999999


In [ ]:

clf = RandomForest(n_trees=30, max_depth=30, min_samples_split=2, n_feature=26)
clf.fit(X_train, y_train)
predictions = clf.predict(X_test)

acc =  accuracy(y_test, predictions)
print(acc)

0.915


### no pca
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 10, 'n_trees': 30} mean_acc=0.9225 std=0.0179 time=172.3s
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 10, 'n_trees': 60} mean_acc=0.9287 std=0.0161 time=349.4s
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 10, 'n_trees': 100} mean_acc=0.9287 std=0.0166 time=596.1s
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 20, 'n_trees': 30} mean_acc=0.9250 std=0.0220 time=365.5s
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 20, 'n_trees': 60} mean_acc=0.9287 std=0.0184 time=770.8s
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 20, 'n_trees': 100} mean_acc=0.9313 std=0.0230 time=1301.8s
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 30, 'n_trees': 30} mean_acc=0.9175 std=0.0222 time=571.6s
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 30, 'n_trees': 60} mean_acc=0.9212 std=0.0146 time=1151.4s
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 30, 'n_trees': 100} mean_acc=0.9275 std=0.0222 time=1998.3s
- params={'max_depth': 10, 'min_samples_split': 2, 'n_features': 40, 'n_trees': 30} mean_acc=0.9163 std=0.0179 time=802.5s